# Day 3: Plan Before You Build

AI agents don't give the exact same answer twice. So how do we test them? We write a **plan** first, then check the agent's output **against the plan**, instead of just asking "does it look right?"

Today we follow four steps:

| Step | What we do |
|---|---|
| **1. Plan** | Write down what goes in, what comes out, the rules, and what "done" means |
| **2. Build** | Create the skeleton, using fake (stub) steps first |
| **3. Fill in** | Replace the stubs with the real LLM logic |
| **4. Check** | Test the result against the plan |

**Our example:** an agent that turns messy meeting notes into a clean summary with action items.

### Before you start

Create a file called `.env` in the same folder as this notebook and add your Azure OpenAI details:

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
AZURE_OPENAI_API_VERSION=2024-10-21
```

Never share this file or upload it to GitHub, because it contains your secret key.

### Setup
This cell loads your `.env` values and creates the `llm` object we use in the rest of this notebook. Run it first.

In [ ]:
# pip install -U langchain langgraph langchain-openai python-dotenv

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# 1. Load the values from the .env file
load_dotenv()

# 2. Make sure nothing important is missing
required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(f"Please add these to your .env file: {missing}")

# 3. Create the LLM
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

print("Setup done. Using deployment:", os.environ["AZURE_OPENAI_DEPLOYMENT"])

In [ ]:
# Quick check: can we talk to the model?
print(llm.invoke("Say hello to the class in one short sentence.").content)

## Step 1: Plan what goes in and what comes out

We write the plan as **Pydantic classes**. A Pydantic class is both a plan a person can read **and** a rule the computer enforces.

- `MeetingInput` describes what the agent accepts.
- `MeetingSummary` describes exactly what the agent must return.

In [ ]:
from pydantic import BaseModel, Field

class MeetingInput(BaseModel):
    notes: str = Field(min_length=50)     # notes must be at least 50 characters
    attendees: list[str]

class ActionItem(BaseModel):
    owner: str
    task: str

class MeetingSummary(BaseModel):
    title: str
    summary: str = Field(description="Short summary, max 60 words")
    decisions: list[str]
    action_items: list[ActionItem]

print("Plan (schemas) ready.")

### The schema catches bad input automatically

We never wrote an `if` check for short notes. Pydantic rejects them because the plan said `min_length=50`.

In [ ]:
from pydantic import ValidationError

try:
    MeetingInput(notes="too short", attendees=["Priya"])
except ValidationError as e:
    print("Rejected:", e.errors()[0]["msg"])

## Step 1 (continued): Rules and the definition of "done"

- **Rules** are things the agent must **never** do. We tell the model these rules in the prompt.
- **Definition of done** is a function that checks the output. If it returns an empty list, the task is finished.

This check is ordinary Python, so the model can't ignore it.

In [ ]:
RULES = """- NEVER give an action item to someone who is not an attendee.
- NEVER invent decisions that are not in the notes."""

def check_against_plan(meeting, result):
    """Definition of DONE: returns a list of problems. Empty list = done."""
    problems = []
    if len(result.action_items) == 0:
        problems.append("No action items found.")
    for item in result.action_items:
        if item.owner not in meeting.attendees:
            problems.append(f"{item.owner} is not an attendee.")
    if len(result.summary.split()) > 60:
        problems.append("Summary is longer than 60 words.")
    return problems

## Step 2: Build the skeleton, using a stub

We build the whole graph **before** using the LLM. The summarize step is a **stub**, a fake function that returns a dummy answer. This lets us test that the wiring works without spending any model calls.

The graph looks like this:

```
summarize  ->  check  ->  done?  yes -> END
                 ^          |
                 |__ retry _| (max 3 attempts)
```

If the check finds problems, the graph loops back and tries again, and it passes the problems along as feedback.

In [ ]:
class SummaryState(TypedDict):
    meeting: MeetingInput
    result: MeetingSummary
    problems: list
    attempts: int


def check_node(state: SummaryState):
    problems = check_against_plan(state["meeting"], state["result"])
    print(f"   Attempt {state['attempts']} -> problems: {problems or 'none'}")
    return {"problems": problems}


def decide_next(state: SummaryState):
    if not state["problems"]:
        return "done"
    if state["attempts"] >= 3:
        return "give_up"
    return "retry"


def build_app(summarize_node):
    graph = StateGraph(SummaryState)
    graph.add_node("summarize", summarize_node)
    graph.add_node("check", check_node)
    graph.add_edge(START, "summarize")
    graph.add_edge("summarize", "check")
    graph.add_conditional_edges("check", decide_next,
                                {"done": END, "give_up": END, "retry": "summarize"})
    return graph.compile()


# STUB: returns a fake answer so we can test the wiring before using the LLM
def stub_summarize(state: SummaryState):
    fake = MeetingSummary(title="stub", summary="stub", decisions=[], action_items=[])
    return {"result": fake, "attempts": state["attempts"] + 1}


NOTES = """Product sync, 12 Sep. Attendees: Priya, Rahul, Anita.
Priya said the mobile app login bug is still open; Rahul will fix it by Friday.
The team decided to delay the v2 launch to 1 October.
Anita will prepare the customer email announcing the new date."""
meeting = MeetingInput(notes=NOTES, attendees=["Priya", "Rahul", "Anita"])

stub_app = build_app(stub_summarize)
stub_app.invoke({"meeting": meeting, "problems": [], "attempts": 0})
print("Wiring works. The stub fails the check, as expected.")

The stub failed 3 times and then stopped. That's exactly what we wanted: the loop, the check, and the attempt limit all work.

## Step 3: Fill in the real LLM step

Now we replace the stub with the real thing. The key line is `with_structured_output(MeetingSummary)`. It **forces** the model to reply in exactly the shape of our plan, so it can't return random text.

In [ ]:
summarizer = llm.with_structured_output(MeetingSummary)

def real_summarize(state: SummaryState):
    m = state["meeting"]
    feedback = "\n".join(state["problems"])
    prompt = f"""Summarize these meeting notes.
Rules:
{RULES}
Attendees: {', '.join(m.attendees)}
Problems from your last attempt (fix them): {feedback or 'none'}

Notes:
{m.notes}"""
    return {"result": summarizer.invoke(prompt), "attempts": state["attempts"] + 1}

app = build_app(real_summarize)   # same skeleton, real step plugged in

## Step 4: Check the output against the plan

First we run the normal notes. Then we try a **tricky** case: Vikram is mentioned in the notes but he is **not** an attendee. The rules say he must not get an action item.

In [ ]:
output = app.invoke({"meeting": meeting, "problems": [], "attempts": 0})
print(output["result"].model_dump_json(indent=2))
print("DONE" if not output["problems"] else "NOT DONE")

In [ ]:
tricky = MeetingInput(
    notes="Budget call. Attendees: Priya, Rahul. Vikram from the vendor will send quotes. "
          "Priya will compare vendor prices next week. We decided to cap spend at 5 lakh.",
    attendees=["Priya", "Rahul"],
)
output2 = app.invoke({"meeting": tricky, "problems": [], "attempts": 0})

for item in output2["result"].action_items:
    print(item.owner, "->", item.task)
print("DONE" if not output2["problems"] else "NOT DONE")

## Key takeaways

- **A plan makes an unpredictable system testable.** We check against rules, not against gut feeling.
- **Schemas are plans the computer enforces.** Bad input is rejected, and the output always has the right shape.
- **Build with stubs first**, so you can test the wiring before you spend model calls.
- **The check plus retry loop** lets the agent fix its own mistakes, using the problem list as feedback.

**Try this:** Add a new rule to `check_against_plan`, for example "every task must start with a verb." Run it again and watch the retry loop use the feedback.